# Higgs, full graph: α and attention-radius sweeps (transductive)

Notebook 04 trained on 500-node METIS subgraphs, which keep only 32% of the follow edges, and no graph model beat
the MLP. A logistic-regression check showed the neighbour signal survives on the full graph but not inside parts.
Here training is **transductive on the whole graph** (`data.sampler: full`, `configs/social/higgs_full.yaml`): the
METIS parts only decide which nodes are labelled in train / val / test (the same nodes as notebook 04 for a given
split seed); message passing and attention run over all 35,593 nodes and 528,532 edges. One optimizer step per
epoch (full batch), up to 500 epochs with patience 100.

Two sweeps, both targets (**`active_after`**, **`activate_passive`**) × 3 split seeds × 2 model seeds, local branch
`gcn`, affine-free branch norms:

1. `configs/sweeps/higgs_full_affine.yaml` (exp `higgs_full_affine`): α ∈ {0, 0.25, 0.5, 0.75, 1} with
   Performer attention, plus `mlp` and `gcn` baselines (84 runs).
2. `configs/sweeps/higgs_full_radius.yaml` (exp `higgs_full_radius`): attention radius k ∈ {0, 1, ∞} at α = 0.5
   (k = 1 is exact softmax over the 1.09M one-hop pairs, k = ∞ is Performer), plus `mlp` and `gcn` (60 runs).
   **k = 2 is not run**: Higgs-follow has 299M two-hop pairs, far beyond a T4 (the run would stop with an
   `attn_max_pairs` error).

**Use a GPU runtime.** Both sweep cells resume: after a disconnect rerun every cell from the top, then the sweep
cells; finished runs are skipped. Results go to `{DRIVE}/results/higgs_full/`.

## 1. Settings (edit)

`BRANCH` is the git branch whose code runs.

In [ ]:
# EDIT: settings for this notebook
BRANCH = 'social-pipeline'          # git branch to run; change it to test a feature branch
USE_TOKEN = False        # True if the repo is private (needs the GITHUB_TOKEN Colab secret, see step 3)
DRIVE = '/content/drive/MyDrive/Thesis/gnn_mech'
DATA_ROOT = f'{DRIVE}/data'    # downloads and processed caches (kept on Drive, reused by every session)
OUT_DIR = f'{DRIVE}/runs'      # run folders (kept on Drive so runs resume after a disconnect)
LOCAL_DATA = '/content/data'   # fast local copy of the dataset used during training
DATASET = 'higgs-follow'
EXTRA = ''               # optional extra overrides for every run, e.g. 'train.epochs=300'

## 2. Check the GPU (Runtime > Change runtime type > T4 GPU)

In [ ]:
!nvidia-smi

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Get the code

Clones the repo at `BRANCH`, or pulls if it is already there. If the repo is private, create a GitHub
personal access token (read access to the repo is enough), store it as a Colab secret named
`GITHUB_TOKEN` (key icon in the left sidebar) and set `USE_TOKEN = True` in step 1. Never paste the
token into the notebook itself.

In [ ]:
import os, subprocess

REPO_DIR = '/content/message-passing-in-gnns'
url = 'https://github.com/djdhawal/message-passing-in-gnns.git'
if USE_TOKEN:
    from google.colab import userdata
    url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/djdhawal/message-passing-in-gnns.git"

if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, url, REPO_DIR], check=True)
%cd {REPO_DIR}
!git log -1 --oneline

## 5. Install

In [ ]:
!pip install -q torch_geometric
!pip install -q -e .

## 6. Folders on Drive

In [ ]:
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(LOCAL_DATA, exist_ok=True)
print('data on Drive:', DATA_ROOT)
print('runs on Drive:', OUT_DIR)

### Sweep files

In [ ]:
SWEEP_AFFINE = 'configs/sweeps/higgs_full_affine.yaml'
SWEEP_RADIUS = 'configs/sweeps/higgs_full_radius.yaml'
SWEEPS = [SWEEP_AFFINE, SWEEP_RADIUS]
for s in SWEEPS:
    print('sweep:', s, '' if os.path.exists(s) else '(MISSING: check BRANCH)')

## 7. Data helpers

`find_source_dir` finds the dataset's folder under `<root>/social/`. `sync` copies only files that are
missing or newer (rsync). Nothing to edit here.

In [ ]:
import shutil
from pathlib import Path


def _source_candidates(name):
    # Folder names under <root>/social/ that may hold `name`; gnn_mech.social.sources decides when it can.
    cands = []
    try:
        from gnn_mech.social import sources
        for attr in ('source_of', 'source_name', 'source_for'):
            fn = getattr(sources, attr, None)
            if callable(fn):
                try:
                    cands.append(str(fn(name)))
                except Exception:
                    pass
        for attr in ('SOURCES', 'SOURCE_OF', 'DATASET_SOURCE', 'DATASETS'):
            table = getattr(sources, attr, None)
            if isinstance(table, dict) and name in table:
                v = table[name]
                if not isinstance(v, str):
                    v = v.get('source') if isinstance(v, dict) else getattr(v, 'source', None)
                if v:
                    cands.append(str(v))
    except Exception:
        pass
    cands.append(name)
    if name.startswith('twitch-') and name != 'twitch-gamers':
        cands += ['musae', 'twitch', 'twitch-musae', 'musae-twitch']
    if name.startswith('higgs-'):
        cands += ['higgs', 'higgs-twitter']
    if name.startswith('reddit-'):
        cands += ['reddit', 'reddit-hyperlinks', 'soc-redditHyperlinks']
    return list(dict.fromkeys(cands))


def find_source_dir(name, root, fetched=None):
    # Name of the folder <root>/social/<source>/ that holds `name`, or None if it does not exist yet.
    social = Path(root) / 'social'
    if isinstance(fetched, (str, os.PathLike)):
        try:
            rel = Path(fetched).resolve().relative_to(social.resolve())
            if rel.parts:
                return rel.parts[0]
        except Exception:
            pass
    for c in _source_candidates(name):
        if (social / c).is_dir():
            return c
    return None


def sync(src, dst):
    # Copy files that are missing or newer in src into dst; never deletes anything.
    os.makedirs(dst, exist_ok=True)
    if shutil.which('rsync'):
        subprocess.run(['rsync', '-a', '--update', f'{src}/', f'{dst}/'], check=True)
    else:
        shutil.copytree(src, dst, dirs_exist_ok=True)

## 8. Download the dataset into Drive (once)

`gnn_mech.social.sources.fetch` downloads and parses the raw files into `DATA_ROOT/social/<source>/` and
skips files that already exist, so later sessions reuse them. If it fails, the run downloads the data
itself on first use.

In [ ]:
fetched = None
try:
    from gnn_mech.social import sources
    fetched = sources.fetch(DATASET, root=DATA_ROOT)
except Exception as e:
    print(f'[warn] fetch({DATASET!r}) failed: {e!r}')
SOURCE = find_source_dir(DATASET, DATA_ROOT, fetched)
print('source folder on Drive:', SOURCE)

## 9. Copy the dataset to local disk

Training reads from `LOCAL_DATA` (much faster than Drive). Only missing or newer files are copied.
Run outputs still go to `OUT_DIR` on Drive.

In [ ]:
if SOURCE:
    sync(f'{DATA_ROOT}/social/{SOURCE}', f'{LOCAL_DATA}/social/{SOURCE}')
    RUN_DATA = LOCAL_DATA
else:
    print('[warn] no dataset folder on Drive yet: this session reads and writes data on Drive directly')
    RUN_DATA = DATA_ROOT
COMMON = f'out_dir={OUT_DIR} data.root={RUN_DATA}'
print(COMMON)

In [ ]:
!du -sh {RUN_DATA}/social/* 2>/dev/null || echo 'nothing local yet'

## 10. Build the full-graph cache (first session only) and save it to Drive

Loads every distinct data config of both sweeps once (2 targets × 3 split seeds = 6 caches). The first time, this
partitions the graph to choose the labelled nodes and writes `social/<source>/processed/<hash>.pt` (a few seconds
each); later sessions find it. Each line prints the graph size and the labelled nodes per split. New files are
then copied back to Drive.

In [ ]:
import gc, json
from gnn_mech.data import load_dataset
from gnn_mech.sweep import load_sweep

DATA_CFGS = []
for s in SWEEPS:
    try:
        DATA_CFGS += [(r.name, r.cfg) for r in load_sweep(s, [f'out_dir={OUT_DIR}', f'data.root={RUN_DATA}'] + ([EXTRA] if EXTRA else []))[0]]
    except Exception as e:
        print(f'[warn] could not read {s}: {e!r}')
seen = set()
for label, c in DATA_CFGS:
    d = c.to_dict()['data']
    d.pop('root', None)
    key = json.dumps(d, sort_keys=True, default=str)
    if key in seen:
        continue
    seen.add(key)
    try:
        tr, va, te, info = load_dataset(c.data)
        g = tr[0]
        lab = ' / '.join(str(int(ds[0].y_mask.sum())) for ds in (tr, va, te))
        print(f'{c.data.target} split {c.data.split_seed}: {g.num_nodes:,} nodes, {g.edge_index.size(1) // 2:,} edges, '
              f'labelled train / val / test {lab}, task {info.task}, feat_dim {getattr(info, "feat_dim", "?")}')
        del tr, va, te, g
    except Exception as e:
        print(f'[warn] could not prepare data for {label}: {e!r} (the run will try again)')
gc.collect()

if SOURCE is None:
    SOURCE = find_source_dir(DATASET, RUN_DATA)
if RUN_DATA != DATA_ROOT and SOURCE:
    sync(f'{RUN_DATA}/social/{SOURCE}', f'{DATA_ROOT}/social/{SOURCE}')
    print('cache saved to Drive:', f'{DATA_ROOT}/social/{SOURCE}')

## 11. α sweep: dry run (run list, status, estimated remaining hours)

In [ ]:
!python -m gnn_mech.sweep {SWEEP_AFFINE} --dry-run {COMMON} {EXTRA}

## 12. α sweep: run (rerun this cell after a disconnect to resume)

In [ ]:
!python -m gnn_mech.sweep {SWEEP_AFFINE} {COMMON} {EXTRA}

## 13. Radius sweep: dry run

In [ ]:
!python -m gnn_mech.sweep {SWEEP_RADIUS} --dry-run {COMMON} {EXTRA}

## 14. Radius sweep: run (rerun this cell after a disconnect to resume)

In [ ]:
!python -m gnn_mech.sweep {SWEEP_RADIUS} {COMMON} {EXTRA}

### Save any new data files back to Drive

In [ ]:
if RUN_DATA != DATA_ROOT:
    SOURCE = SOURCE or find_source_dir(DATASET, RUN_DATA)
    if SOURCE:
        sync(f'{RUN_DATA}/social/{SOURCE}', f'{DATA_ROOT}/social/{SOURCE}')
        print('synced to Drive')

## 15. Results

Test metric per target, mean ± std over split seeds × model seeds; `mlp` and `gcn` as dashed lines (shaded ± std). Tables and figures are saved to `{DRIVE}/results/higgs_full/`.

### Collection helpers

In [ ]:
import json, os
import numpy as np
import pandas as pd

PRIMARY = ('auroc', 'acc', 'r2', 'ap', 'mae', 'rmse')   # first one present is the headline metric
BASELINES = ('mlp', 'gcn')


def read_json(path):
    try:
        with open(path) as f:
            return json.load(f)
    except Exception:
        return None


def primary_metric(split):
    keys = [k for k, v in (split or {}).items() if k != 'loss' and isinstance(v, (int, float))]
    for k in PRIMARY:
        if k in keys:
            return k
    return keys[0] if keys else None

def collect(out_dir, exps=None):
    # One row per finished run (config.json + metrics.json); unreadable or unfinished runs are skipped.
    rows = []
    root = Path(out_dir)
    if not root.is_dir():
        return pd.DataFrame()
    for cfg_path in root.glob('*/*/seed*/config.json'):
        run = cfg_path.parent
        exp = run.parent.parent.name
        if exps is not None and exp not in exps:
            continue
        cfg, m = read_json(cfg_path), read_json(run / 'metrics.json')
        if not cfg or not m or not isinstance(m.get('test'), dict):
            continue
        model, data = cfg.get('model', {}), cfg.get('data', {})
        metric = primary_metric(m['test'])
        row = {'exp': exp, 'hash': run.parent.name, 'dataset': data.get('name'), 'target': data.get('target', ''),
               'arch': model.get('arch'), 'attn': model.get('attn', 'softmax'), 'alpha': model.get('alpha'),
               'seed': cfg.get('seed'), 'metric': metric,
               'test': m['test'].get(metric), 'val': (m.get('val') or {}).get(metric),
               'best_epoch': m.get('best_epoch'), 'epochs_run': m.get('epochs_run'),
               'hours': (m.get('seconds') or 0) / 3600, 'n_params': m.get('n_params'), 'path': str(run)}
        row.update({f'test_{k}': v for k, v in m['test'].items() if isinstance(v, (int, float))})
        rows.append(row)
    return pd.DataFrame(rows)



def add_cfg_fields(df):
    # split seed, radius and branch-affine are not in collect(); read them from each run's config.json
    extra = []
    for p in df['path']:
        cfg = read_json(Path(p) / 'config.json') or {}
        m, d = cfg.get('model', {}), cfg.get('data', {})
        extra.append({'split_seed': d.get('split_seed', 0), 'radius': m.get('attn_radius', -1),
                      'halo': d.get('halo_hops', 0), 'local': m.get('local')})
    return pd.concat([df.reset_index(drop=True), pd.DataFrame(extra)], axis=1)


def summarize(df, keys):
    # mean +- std over model seeds x split seeds; n = number of runs
    g = df.groupby(keys, dropna=False)
    out = g.agg(n=('test', 'size'), mean=('test', 'mean'), std=('test', 'std'),
                val_mean=('val', 'mean'), minutes=('hours', lambda h: 60 * h.sum())).reset_index()
    out['std'] = out['std'].fillna(0.0)
    out['test (mean ± std)'] = [f'{a:.4f} ± {b:.4f}' for a, b in zip(out['mean'], out['std'])]
    return out


def radius_label(row):
    # attention radius of a run: k=0 for the alpha-0 MPNN, k=<r> for sparse radius attention, k=inf for Performer
    if row['arch'] != 'hybrid':
        return row['arch']
    if float(row['alpha']) == 0.0:
        return 'k=0'
    if row['attn'] == 'radius' and int(row['radius']) >= 0:
        return f"k={int(row['radius'])}"
    return 'k=inf'


def plot_curves(table, xcol, xorder, xlabel, title, path):
    # one panel per target: hybrid mean +- std over runs vs xcol, mlp / gcn as horizontal lines
    targets = sorted(table['target'].unique())
    fig, axes = plt.subplots(1, len(targets), figsize=(5.5 * len(targets), 3.8), squeeze=False)
    for ax, t in zip(axes[0], targets):
        sub = table[table['target'] == t]
        hyb = sub[sub['arch'] == 'hybrid'].copy()
        hyb['_x'] = [xorder.index(v) if v in xorder else len(xorder) for v in hyb[xcol]]
        hyb = hyb.sort_values('_x')
        ax.errorbar(hyb['_x'], hyb['mean'], yerr=hyb['std'], marker='o', capsize=3, label='hybrid (gcn local)')
        for i, b in enumerate(BASELINES):
            r = sub[sub['arch'] == b]
            if len(r):
                m, s = r['mean'].iloc[0], r['std'].iloc[0]
                ax.axhline(m, ls='--', color=f'C{i + 1}', label=f'{b} ({m:.4f})')
                ax.axhspan(m - s, m + s, color=f'C{i + 1}', alpha=0.12)
        ax.set_xticks(range(len(xorder)), xorder)
        ax.set(title=f'{title}: {t}', xlabel=xlabel, ylabel=f"test {sub['metric'].iloc[0]}")
        ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.show()

### α sweep (`higgs_full_affine`)

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from gnn_mech.sweep import load_sweep

RESULTS = f'{DRIVE}/results/higgs_full'
os.makedirs(RESULTS, exist_ok=True)
aff = collect(OUT_DIR, ['higgs_full_affine'])
if aff.empty:
    print('no finished higgs_full_affine runs yet')
else:
    aff = add_cfg_fields(aff)
    aff.to_csv(f'{RESULTS}/affine_runs.csv', index=False)
    aff['variant'] = np.where(aff['arch'] == 'hybrid', 'alpha ' + aff['alpha'].astype(float).map('{:g}'.format), aff['arch'])
    aff_table = summarize(aff, ['target', 'metric', 'variant', 'arch', 'alpha'])
    aff_table.to_csv(f'{RESULTS}/affine_summary.csv', index=False)
    display(aff_table.sort_values(['target', 'arch', 'alpha'])[['target', 'metric', 'variant', 'n', 'test (mean ± std)', 'val_mean', 'minutes']])
    ALPHAS = ['alpha 0', 'alpha 0.25', 'alpha 0.5', 'alpha 0.75', 'alpha 1']
    plot_curves(aff_table, 'variant', ALPHAS, 'alpha (0 = MPNN, 1 = Performer attention)', 'full graph',
                f'{RESULTS}/affine_alpha_curves.png')

### Radius sweep (`higgs_full_radius`)

In [ ]:
rad = collect(OUT_DIR, ['higgs_full_radius'])
if rad.empty:
    print('no finished higgs_full_radius runs yet')
else:
    rad = add_cfg_fields(rad)
    rad['variant'] = rad.apply(radius_label, axis=1)
    rad.to_csv(f'{RESULTS}/radius_runs.csv', index=False)
    rad_table = summarize(rad, ['target', 'metric', 'variant', 'arch'])
    rad_table.to_csv(f'{RESULTS}/radius_summary.csv', index=False)
    display(rad_table.sort_values(['target', 'arch', 'variant'])[['target', 'metric', 'variant', 'n', 'test (mean ± std)', 'val_mean', 'minutes']])
    KS = ['k=0', 'k=1', 'k=2', 'k=inf']
    KS = [k for k in KS if k in set(rad_table['variant'])]
    plot_curves(rad_table, 'variant', KS, 'attention radius k (alpha 0.5; k=0 is the MPNN)', 'full graph',
                f'{RESULTS}/radius_curves.png')

### Headline: does the graph model beat the MLP on the full graph?

In [ ]:
# Headline: best hybrid variant vs the baselines, per sweep and target (positive gap = graph model ahead of the MLP)
rows = []
for name, df in (('affine', aff), ('radius', rad)):
    if df.empty:
        continue
    t = summarize(df, ['target', 'metric', 'variant', 'arch'])
    for tgt, sub in t.groupby('target'):
        hyb = sub[sub['arch'] == 'hybrid']
        if hyb.empty:
            continue
        best = hyb.loc[hyb['mean'].idxmax()]
        base = {b: sub.loc[sub['arch'] == b, 'mean'] for b in BASELINES}
        rows.append({'sweep': name, 'target': tgt, 'metric': best['metric'], 'best hybrid': best['variant'],
                     'best mean': best['mean'],
                     **{f'{b} mean': (v.iloc[0] if len(v) else np.nan) for b, v in base.items()},
                     'best - mlp': best['mean'] - (base['mlp'].iloc[0] if len(base['mlp']) else np.nan)})
headline = pd.DataFrame(rows)
if len(headline):
    headline.to_csv(f'{RESULTS}/headline.csv', index=False)
    display(headline.round(4))
print('saved to', RESULTS)